# Sesión 3 — Comparación de métodos heurísticos y metaheurísticos

En este notebook resolveremos un mismo problema de ruteo mediante **tres métodos diferentes**. La intención no es repetir la teoría de la presentación, sino observar cómo cambia la calidad de la solución y el esfuerzo computacional según el algoritmo utilizado.

Métodos:
1. **Vecino más cercano** — heurística constructiva.
2. **2-opt** — búsqueda local.
3. **Recocido simulado** — metaheurística.

## 1. Librerías

Utilizaremos `numpy`, `pandas` y `matplotlib`. Para este ejemplo no se requiere instalar una librería especializada de optimización.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import math
import random

# Parte 1 — Problema de ruteo

Un vehículo debe salir del CEDIS, visitar todos los clientes una sola vez y regresar al punto de origen. Usaremos coordenadas ficticias para representar las ubicaciones.

In [ ]:
puntos = {
    "CEDIS": (5, 5),
    "A": (2, 8),
    "B": (4, 9),
    "C": (8, 9),
    "D": (9, 6),
    "E": (8, 3),
    "F": (6, 1),
    "G": (3, 2),
    "H": (1, 4),
    "I": (2, 6),
    "J": (6, 7),
    "K": (7, 5)
}

nombres = list(puntos.keys())
coordenadas = np.array([puntos[n] for n in nombres], dtype=float)

pd.DataFrame(
    coordenadas,
    index=nombres,
    columns=["X", "Y"]
)

## 2. Visualización de los puntos

El CEDIS será el punto inicial y final de todas las rutas.

In [ ]:
plt.figure(figsize=(8, 7))

plt.scatter(coordenadas[:, 0], coordenadas[:, 1], s=120)

for nombre, (x, y) in puntos.items():
    plt.text(x + 0.12, y + 0.12, nombre, fontsize=10)

plt.title("CEDIS y clientes")
plt.xlabel("Coordenada X")
plt.ylabel("Coordenada Y")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Matriz de distancias

Calcularemos la distancia euclidiana entre todos los pares de ubicaciones. Esta matriz será utilizada por los tres métodos.

In [ ]:
def distancia(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

n = len(nombres)
matriz_distancias = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        matriz_distancias[i, j] = distancia(
            coordenadas[i],
            coordenadas[j]
        )

df_distancias = pd.DataFrame(
    matriz_distancias,
    index=nombres,
    columns=nombres
)

df_distancias.round(2)

## 4. Funciones auxiliares

Estas funciones permitirán calcular la distancia total y visualizar cualquier ruta obtenida.

In [ ]:
def distancia_ruta(ruta):
    total = 0

    for i in range(len(ruta) - 1):
        total += matriz_distancias[ruta[i], ruta[i + 1]]

    return total


def mostrar_ruta(ruta, titulo):
    plt.figure(figsize=(8, 7))

    for i in range(len(ruta) - 1):
        a = coordenadas[ruta[i]]
        b = coordenadas[ruta[i + 1]]

        plt.plot(
            [a[0], b[0]],
            [a[1], b[1]],
            linewidth=2
        )

    plt.scatter(coordenadas[:, 0], coordenadas[:, 1], s=120)

    for nombre, (x, y) in puntos.items():
        plt.text(x + 0.12, y + 0.12, nombre, fontsize=10)

    plt.title(titulo)
    plt.xlabel("Coordenada X")
    plt.ylabel("Coordenada Y")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def nombres_ruta(ruta):
    return " → ".join(nombres[i] for i in ruta)

# Parte 2 — Método 1: Vecino más cercano

La ruta comienza en el CEDIS y, en cada paso, selecciona el cliente no visitado que se encuentra más cerca de la posición actual.

In [ ]:
def vecino_mas_cercano(inicio=0):
    no_visitados = set(range(len(nombres)))
    no_visitados.remove(inicio)

    ruta = [inicio]
    actual = inicio

    while no_visitados:
        siguiente = min(
            no_visitados,
            key=lambda j: matriz_distancias[actual, j]
        )

        ruta.append(siguiente)
        no_visitados.remove(siguiente)
        actual = siguiente

    ruta.append(inicio)

    return ruta


inicio = time.perf_counter()
ruta_vecino = vecino_mas_cercano()
tiempo_vecino = time.perf_counter() - inicio

dist_vecino = distancia_ruta(ruta_vecino)

print("Ruta:")
print(nombres_ruta(ruta_vecino))
print(f"Distancia total: {dist_vecino:.2f}")
print(f"Tiempo: {tiempo_vecino:.6f} segundos")

In [ ]:
mostrar_ruta(
    ruta_vecino,
    f"Vecino más cercano | Distancia = {dist_vecino:.2f}"
)

# Parte 3 — Método 2: Búsqueda local 2-opt

Partiremos de la ruta generada por vecino más cercano. El método intercambia segmentos de la ruta y conserva los cambios que reduzcan la distancia total.

In [ ]:
def dos_opt(ruta):
    mejor_ruta = ruta.copy()
    mejor_distancia = distancia_ruta(mejor_ruta)

    mejora = True

    while mejora:
        mejora = False

        for i in range(1, len(mejor_ruta) - 2):
            for j in range(i + 1, len(mejor_ruta) - 1):

                candidata = (
                    mejor_ruta[:i]
                    + mejor_ruta[i:j + 1][::-1]
                    + mejor_ruta[j + 1:]
                )

                distancia_candidata = distancia_ruta(candidata)

                if distancia_candidata < mejor_distancia - 1e-9:
                    mejor_ruta = candidata
                    mejor_distancia = distancia_candidata
                    mejora = True
                    break

            if mejora:
                break

    return mejor_ruta


inicio = time.perf_counter()
ruta_2opt = dos_opt(ruta_vecino)
tiempo_2opt = time.perf_counter() - inicio

dist_2opt = distancia_ruta(ruta_2opt)

print("Ruta:")
print(nombres_ruta(ruta_2opt))
print(f"Distancia total: {dist_2opt:.2f}")
print(f"Tiempo: {tiempo_2opt:.6f} segundos")

In [ ]:
mostrar_ruta(
    ruta_2opt,
    f"2-opt | Distancia = {dist_2opt:.2f}"
)

# Parte 4 — Método 3: Recocido simulado

Ahora utilizaremos una metaheurística. A diferencia de la búsqueda local, el recocido simulado puede aceptar temporalmente soluciones peores para explorar otras regiones del espacio de soluciones.

In [ ]:
def recocido_simulado(
    ruta_inicial,
    temperatura_inicial=100,
    temperatura_final=0.001,
    enfriamiento=0.995,
    iteraciones_por_temperatura=30,
    semilla=42
):
    random.seed(semilla)

    actual = ruta_inicial.copy()
    costo_actual = distancia_ruta(actual)

    mejor = actual.copy()
    mejor_costo = costo_actual

    temperatura = temperatura_inicial
    historial = [mejor_costo]

    while temperatura > temperatura_final:

        for _ in range(iteraciones_por_temperatura):

            i, j = sorted(
                random.sample(
                    range(1, len(actual) - 1),
                    2
                )
            )

            candidata = (
                actual[:i]
                + actual[i:j + 1][::-1]
                + actual[j + 1:]
            )

            costo_candidata = distancia_ruta(candidata)
            diferencia = costo_candidata - costo_actual

            if diferencia < 0:
                aceptar = True
            else:
                probabilidad = math.exp(-diferencia / temperatura)
                aceptar = random.random() < probabilidad

            if aceptar:
                actual = candidata
                costo_actual = costo_candidata

            if costo_actual < mejor_costo:
                mejor = actual.copy()
                mejor_costo = costo_actual

            historial.append(mejor_costo)

        temperatura *= enfriamiento

    return mejor, historial


inicio = time.perf_counter()

ruta_sa, historial_sa = recocido_simulado(
    ruta_vecino
)

tiempo_sa = time.perf_counter() - inicio
dist_sa = distancia_ruta(ruta_sa)

print("Ruta:")
print(nombres_ruta(ruta_sa))
print(f"Distancia total: {dist_sa:.2f}")
print(f"Tiempo: {tiempo_sa:.6f} segundos")

In [ ]:
mostrar_ruta(
    ruta_sa,
    f"Recocido simulado | Distancia = {dist_sa:.2f}"
)

## 5. Evolución de la metaheurística

El historial permite observar cómo la mejor solución encontrada cambia conforme avanza la búsqueda.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(historial_sa)

plt.xlabel("Iteración")
plt.ylabel("Mejor distancia encontrada")
plt.title("Evolución del recocido simulado")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Parte 5 — Comparación de los tres métodos

Compararemos la distancia obtenida y el tiempo de ejecución. En problemas pequeños las diferencias de tiempo pueden ser mínimas; el interés principal está en observar cómo cada estrategia explora el espacio de soluciones.

In [ ]:
resultados = pd.DataFrame({
    "Método": [
        "Vecino más cercano",
        "2-opt",
        "Recocido simulado"
    ],
    "Tipo": [
        "Heurística constructiva",
        "Búsqueda local",
        "Metaheurística"
    ],
    "Distancia": [
        dist_vecino,
        dist_2opt,
        dist_sa
    ],
    "Tiempo (s)": [
        tiempo_vecino,
        tiempo_2opt,
        tiempo_sa
    ]
})

mejor_distancia = resultados["Distancia"].min()

resultados["Diferencia vs. mejor (%)"] = (
    (resultados["Distancia"] - mejor_distancia)
    / mejor_distancia * 100
)

resultados.round({
    "Distancia": 2,
    "Tiempo (s)": 6,
    "Diferencia vs. mejor (%)": 2
})

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    resultados["Método"],
    resultados["Distancia"]
)

plt.ylabel("Distancia total")
plt.title("Comparación de calidad de solución")
plt.tight_layout()
plt.show()

## 6. ¿Los tres métodos siempre obtienen lo mismo?

Para observar la naturaleza del recocido simulado, ejecutaremos la metaheurística varias veces utilizando diferentes semillas.

In [ ]:
experimentos = []

for semilla in range(1, 11):

    inicio = time.perf_counter()

    ruta_exp, _ = recocido_simulado(
        ruta_vecino,
        semilla=semilla
    )

    tiempo_exp = time.perf_counter() - inicio

    experimentos.append({
        "Semilla": semilla,
        "Distancia": distancia_ruta(ruta_exp),
        "Tiempo (s)": tiempo_exp,
        "Ruta": nombres_ruta(ruta_exp)
    })

df_experimentos = pd.DataFrame(experimentos)
df_experimentos.round({
    "Distancia": 2,
    "Tiempo (s)": 6
})

### Preguntas para discusión

- ¿Cuál método encontró la ruta de menor distancia?
- ¿Cuál obtuvo una solución más rápidamente?
- ¿2-opt mejoró la solución inicial del vecino más cercano?
- ¿Por qué el recocido simulado puede aceptar temporalmente una ruta peor?
- ¿Esperaría las mismas diferencias si el problema tuviera 100 o 500 clientes?

# Parte 6 — Ejercicio para el alumno

Agregue cinco clientes adicionales al problema y vuelva a ejecutar los tres métodos.

Compare:
1. La distancia obtenida.
2. El tiempo de ejecución.
3. La diferencia porcentual respecto a la mejor solución encontrada.
4. La estabilidad del recocido simulado al ejecutarlo varias veces.

Finalmente, explique qué método utilizaría si tuviera que generar rutas diariamente para una red de cientos de clientes.

# Cierre

Los tres algoritmos resuelven el mismo problema, pero utilizan estrategias diferentes. **Vecino más cercano** construye rápidamente una solución; **2-opt** intenta mejorarla mediante cambios locales; y **recocido simulado** amplía la exploración permitiendo movimientos que temporalmente empeoran la solución.

Esta comparación permite observar la relación entre **calidad de solución, tiempo computacional y estrategia de búsqueda**.